# 02 Descriptives

Descriptive tables for Memo 1. Writes CSVs to outputs/tables/.
Descriptive figures for Memo 1. Reads outputs/tables, writes PNGs to outputs/figures/.

In [ ]:
%run 00_utils.ipynb

import pandas as pd, matplotlib.pyplot as plt

set_style(font_size=12)

## Tables

In [ ]:
pqr = load_clean("pqr_medicines")
pqr["manufacturer"] = pqr["firm"]   # harmonized firm names (see firm_crosswalk.csv)
fpp = load_clean("pq_fpp")
api = load_clean("pq_api")

In [ ]:
# T1 dataset overview
overview_rows = []
for name, table, unit, firm_column, product_column, year_column in [
    ("Global Fund PQR (medicines)", pqr, "transaction", "manufacturer", "product", "year"),
    ("WHO PQ finished products", fpp, "prequalified product", "firm", "INN, Dosage Form and Strength", "year"),
    ("WHO PQ active ingredients", api, "prequalified API", "firm", "INN", "year")]:
    overview_rows.append({"dataset": name, "unit": unit, "n_obs": len(table),
                 "n_firms": table[firm_column].nunique(), "n_products": table[product_column].nunique(),
                 "first_year": int(table[year_column].min()), "last_year": int(table[year_column].max())})
dataset_overview = pd.DataFrame(overview_rows)
dataset_overview.to_csv(TABLES / "t1_dataset_overview.csv", index=False)
print(dataset_overview.to_string(index=False))

PQR gave me 74,689 medicine transactions from 110 firms and 128 products. The WHO lists added 655 finished products and 194 active ingredients. PQR is pooled over 2005-2026, but 2025-2026 are incomplete because of reporting lags.

In [ ]:
# T2 PQR by therapeutic area
area_summary = pqr.groupby("area").agg(transactions=("pk", "size"),
                             spend_usd_m=("cost_usd", lambda x: x.sum() / 1e6),
                             products=("product", "nunique"),
                             manufacturers=("manufacturer", "nunique"),
                             countries=("country", "nunique")).round(1)
area_summary.to_csv(TABLES / "t2_pqr_by_area.csv")
print("\n", area_summary.to_string())

HIV dominated spend, at USD 8.3 bn of 11.7 bn. TB had the most products (60) and manufacturers (62), but its spend was about the same as malaria's. Spend alone would therefore understate how much of the supplier network is in TB.

In [ ]:
# T3 supplier concentration per product (spend-based HHI, 0-10000)
firm_product_spend = pqr.groupby(["area", "product", "manufacturer"])["cost_usd"].sum().reset_index()
def concentration(product_rows):
    spend_share = product_rows["cost_usd"] / product_rows["cost_usd"].sum()
    return pd.Series({"n_manufacturers": len(product_rows), "spend_usd_m": product_rows["cost_usd"].sum() / 1e6,
                      "top1_share": spend_share.max(), "hhi": (spend_share ** 2).sum() * 10000,
                      "top_manufacturer": product_rows.loc[product_rows["cost_usd"].idxmax(), "manufacturer"]})
product_concentration = firm_product_spend.groupby(["area", "product"]).apply(concentration, include_groups=False).reset_index()
product_concentration["vulnerable_le2"] = product_concentration["n_manufacturers"] <= 2
product_concentration = product_concentration.sort_values("spend_usd_m", ascending=False)
product_concentration.round(3).to_csv(TABLES / "t3_product_concentration.csv", index=False)
print("\nproducts:", len(product_concentration), "| <=2 manufacturers:", product_concentration.vulnerable_le2.sum(),
      "| HHI>2500:", (product_concentration.hhi > 2500).sum())
print("spend share in <=2-mfr products:",
      round(product_concentration.loc[product_concentration.vulnerable_le2, "spend_usd_m"].sum() / product_concentration.spend_usd_m.sum(), 3))
print(product_concentration.groupby("area")[["n_manufacturers", "hhi"]].median().round(0).to_string())
print("\nlargest-spend vulnerable products:")
print(product_concentration[product_concentration.vulnerable_le2].head(10)[["area","product","n_manufacturers","spend_usd_m","top_manufacturer"]].round(1).to_string(index=False))

57 of 128 products (45%) had only 1 or 2 manufacturers over the whole period. These were mostly niche or newer products, such as delamanid and pretomanid, and they made up only 2.8% of spend. A low supplier count is common but does not show a problem by itself, since one or two firms may already meet demand. Median HHI was high in all three areas (HIV 5,152; TB 7,327; malaria 7,949). These HHIs pool all years, so a product could look less concentrated than it was in any single year.

In [ ]:
# T4 manufacturer breadth and spend
manufacturers = pqr.groupby("manufacturer").agg(products=("product", "nunique"),
                                    areas=("area", "nunique"),
                                    transactions=("pk", "size"),
                                    spend_usd_m=("cost_usd", lambda x: x.sum() / 1e6),
                                    first_year=("year", "min"), last_year=("year", "max"))
manufacturers["spend_share"] = manufacturers.spend_usd_m / manufacturers.spend_usd_m.sum()
manufacturers = manufacturers.sort_values("spend_usd_m", ascending=False)
manufacturers.round(3).to_csv(TABLES / "t4_manufacturers.csv")
print("\nmanufacturers:", len(manufacturers), "| top5 spend share:", round(manufacturers.spend_share.head(5).sum(), 3),
      "| top10:", round(manufacturers.spend_share.head(10).sum(), 3),
      "| in >=2 areas:", (manufacturers.areas >= 2).sum(), "| all 3 areas:", (manufacturers.areas == 3).sum())
print(manufacturers.head(10)[["products","areas","spend_usd_m","spend_share"]].round(3).to_string())

Supply was concentrated among a few firms. The top 5 held 62% of spend and the top 10 held 79%, and Mylan alone held 25%. Only 5 firms supplied all three disease areas. Guilin was the only Chinese firm in the top 10 (4.3%, malaria only).

In [ ]:
# T5 annual series (for figures)
recent_pqr = pqr[pqr.year.between(2008, 2025)]
annual_by_area = recent_pqr.groupby(["year", "area"]).agg(transactions=("pk", "size"),
                                      spend_usd_m=("cost_usd", lambda x: x.sum() / 1e6),
                                      active_manufacturers=("manufacturer", "nunique")).reset_index()
annual_by_area.round(1).to_csv(TABLES / "t5_pqr_annual_by_area.csv", index=False)
prequal_by_area = fpp.dropna(subset=["year"]).groupby(["year", "Therapeutic Area"]).size().rename("n").reset_index()
prequal_by_area.to_csv(TABLES / "t6_pq_fpp_annual_by_area.csv", index=False)

## Figures

In [ ]:
AREA_COLORS = {**AREA_COLORS, "Other": GRAY}

def title(ax, text, subtitle):
    add_title(ax, text, subtitle, size=14, sub_size=11, pad=24)

def source(fig, note):
    fig.text(0.01, -0.06, note, fontsize=9.5, color=MUTED, ha="left")

In [ ]:
# F1 annual PQR spend by area
annual = pd.read_csv(TABLES / "t5_pqr_annual_by_area.csv")
fig, ax = plt.subplots(figsize=(6.5, 4))
for area in ["HIV/ARV", "Tuberculosis", "Malaria"]:
    area_series = annual[annual.area == area]
    ax.plot(area_series.year, area_series.spend_usd_m, color=AREA_COLORS[area], lw=2, marker="o", ms=3.5, solid_capstyle="round")
    # 2025 end points of TB and malaria sit close together, so spread their labels
    nudge = {"HIV/ARV": 30, "Tuberculosis": -35, "Malaria": 35}[area]
    ax.text(area_series.year.iloc[-1] + 0.3, area_series.spend_usd_m.iloc[-1] + nudge, area, color=INK, fontsize=11, va="center")
ax.set_xlim(2007.5, 2027.3)
ax.set_ylim(0)
ax.grid(axis="x", visible=False)
ax.set_xticks(range(2008, 2026, 2))
ax.set_ylabel("Spend per year (USD million)")
title(ax, "Global Fund medicine spending by disease area, 2008-2025",
      "Total product cost reported to PQR, by purchase order year")
source(fig, "Source: Global Fund Price & Quality Reporting, downloaded 2026-10-10. 2025 may be incomplete due to reporting lags.")
save_figure(fig, "f1_pqr_spend_by_year.png")

HIV transactions peaked in 2013 and then fell as regimens moved to fewer fixed-dose combinations (t5). TB passed HIV in transaction counts in 2023. The last years in the figure should be read with care because of reporting lags.

In [ ]:
# F2 manufacturers per product
concentration_table = pd.read_csv(TABLES / "t3_product_concentration.csv")
supplier_counts = concentration_table.n_manufacturers.clip(upper=12).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6.5, 4))
bar_colors = ["#7C1220" if n_suppliers <= 2 else AREA_COLORS["HIV/ARV"] for n_suppliers in supplier_counts.index]
ax.bar(supplier_counts.index, supplier_counts.values, color=bar_colors, width=0.8, edgecolor="white", linewidth=1)
for n_suppliers, n_products in supplier_counts.items():
    ax.text(n_suppliers, n_products + 0.4, str(n_products), ha="center", fontsize=10.5, color=INK2)
ax.set_xticks(supplier_counts.index, [str(n_suppliers) if n_suppliers < 12 else "12+" for n_suppliers in supplier_counts.index])
ax.set_xlabel("Number of manufacturers that supplied the product")
ax.set_ylabel("Number of products")
ax.grid(axis="x", visible=False)
n_thin = int((concentration_table.n_manufacturers <= 2).sum())
ax.annotate(f"{n_thin} of {len(concentration_table)} products have\n1 or 2 manufacturers", xy=(2.3, supplier_counts.loc[2] + 1),
            xytext=(4.2, supplier_counts.max() * 0.85), fontsize=11, color=INK,
            arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))
title(ax, "Many Global Fund medicines rely on one or two suppliers",
      "Distinct manufacturers per product, all PQR medicine transactions 2005-2026")
source(fig, "Source: Global Fund PQR. Products are PQR product names (formulation level); manufacturers harmonized to firms.")
save_figure(fig, "f2_manufacturers_per_product.png")

In [ ]:
# F3 manufacturer spend concentration
firm_table = pd.read_csv(TABLES / "t4_manufacturers.csv").sort_values("spend_usd_m", ascending=False)
top_firms = firm_table.head(10).copy()
short_names = {"Guilin Pharmaceutical": "Guilin (Fosun)"}
top_firms["label"] = top_firms.manufacturer.map(short_names).fillna(top_firms.manufacturer)
other_firms = pd.DataFrame({"label": [f"Other {len(firm_table) - 10} manufacturers"], "spend_share": [firm_table.spend_share.iloc[10:].sum()]})
shares = pd.concat([top_firms[["label", "spend_share"]], other_firms]).iloc[::-1]
fig, ax = plt.subplots(figsize=(6.5, 4.6))
bar_colors = [AREA_COLORS["Other"] if label.startswith("Other") else AREA_COLORS["HIV/ARV"] for label in shares.label]
ax.barh(shares.label, shares.spend_share * 100, color=bar_colors, height=0.72, edgecolor="white", linewidth=1)
for i, share in enumerate(shares.spend_share * 100):
    ax.text(share + 0.4, i, f"{share:.1f}%", va="center", fontsize=10.5, color=INK2)
ax.set_xlabel("Share of total medicine spend (%)")
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
title(ax, "Ten firms account for three quarters of medicine spend",
      f"Share of USD {firm_table.spend_usd_m.sum()/1000:.1f} bn in PQR medicine purchases, 2005-2026")
source(fig, "Source: Global Fund PQR. Total Product Cost (USD); excludes freight.")
save_figure(fig, "f3_manufacturer_spend_share.png")

In [ ]:
# F4 WHO PQ finished products prequalified per year
prequal = pd.read_csv(TABLES / "t6_pq_fpp_annual_by_area.csv")
prequal["grp"] = prequal["Therapeutic Area"].map({"HIV/AIDS": "HIV/ARV", "Tuberculosis": "Tuberculosis", "Malaria": "Malaria"}).fillna("Other")
prequal_by_year = prequal.pivot_table(index="year", columns="grp", values="n", aggfunc="sum").fillna(0).reindex(range(2002, 2027), fill_value=0)
area_order = ["HIV/ARV", "Tuberculosis", "Malaria", "Other"]
fig, ax = plt.subplots(figsize=(6.5, 4))
bottom = 0
for group in area_order:
    ax.bar(prequal_by_year.index, prequal_by_year[group], bottom=bottom, color=AREA_COLORS[group], width=0.8, edgecolor="white", linewidth=1, label=group)
    bottom = bottom + prequal_by_year[group]
ax.set_xticks(range(2002, 2027, 2))
ax.grid(axis="x", visible=False)
ax.set_ylabel("Products prequalified")
ax.legend(frameon=False, ncol=4, loc="upper left", fontsize=11)
ax.set_ylim(0, bottom.max() * 1.18)
title(ax, "WHO prequalifications of currently listed finished products",
      "By year of prequalification and therapeutic area (n = 564 dated products)")
source(fig, "Source: WHO PQ medicines list, downloaded 2026-10-10. Only products still listed today; earlier years undercount (withdrawn products drop out).")
save_figure(fig, "f4_pq_fpp_by_year.png")

Prequalifications of currently listed products rose from mostly under 15 per year before 2012 to about 40-50 per year in 2018-2024. Newer listings also include areas outside the three diseases, such as reproductive health, oncology and hepatitis. WHO only lists products that are still prequalified, so the early years undercount.

Chinese firms were more visible upstream than downstream. Shanghai Desano had 20 prequalified active ingredients, while Guilin was the only Chinese firm among the top finished-product suppliers in PQR.